In [7]:
# 将xlxs提取为json

import pandas as pd
import json
from openpyxl import load_workbook

def extract_columns_to_json(file_path, columns_to_extract, output_json_path=None):
    """
    从Excel文件中提取指定列并转换为JSON
    按照Excel单元格显示的内容读取，而不是存储的原始值
    
    参数:
    file_path: Excel文件路径
    columns_to_extract: 要提取的列字母列表，如 ['B', 'D', 'K']
    output_json_path: 输出的JSON文件路径（可选）
    
    返回:
    JSON字符串和DataFrame对象
    """
    try:
        # 使用openpyxl读取Excel文件，获取显示值
        wb = load_workbook(file_path, data_only=False)
        ws = wb.active
        
        # 获取列名（从第一行读取）
        header_row = 1
        column_names = []
        column_indices = []
        
        for col_letter in columns_to_extract:
            col_index = ord(col_letter.upper()) - ord('A')
            # 读取第一行作为列名
            cell = ws[f"{col_letter}{header_row}"]
            if cell.value is not None:
                column_names.append(str(cell.value).strip())
                column_indices.append(col_index)
            else:
                print(f"警告: 列 {col_letter} 的第一行为空，跳过")
        
        # 读取数据行（从第二行开始）
        data_rows = []
        for row_idx in range(header_row + 1, ws.max_row + 1):
            row_data = {}
            for i, col_letter in enumerate(columns_to_extract):
                if i < len(column_names):
                    cell = ws[f"{col_letter}{row_idx}"]
                    # 获取单元格的显示值（格式化后的值）
                    if cell.value is None:
                        cell_value = ""
                    elif cell.data_type == 'n':  # 数字类型
                        # 检查是否是日期格式
                        number_format = str(cell.number_format).lower()
                        is_date_format = any(keyword in number_format for keyword in ['date', 'yy', 'mm', 'dd', 'yyyy'])
                        
                        if is_date_format and isinstance(cell.value, (int, float)):
                            # Excel日期是从1900-01-01开始的天数（但有个bug，1900年被当作闰年）
                            # 所以实际基准是1899-12-30
                            try:
                                from datetime import datetime, timedelta
                                # Excel日期序列号转换
                                base_date = datetime(1899, 12, 30)
                                date_value = base_date + timedelta(days=cell.value)
                                # 根据格式字符串格式化日期
                                if '/' in number_format or 'yy' in number_format:
                                    cell_value = date_value.strftime('%Y/%m/%d')
                                else:
                                    cell_value = date_value.strftime('%Y-%m-%d')
                            except Exception as e:
                                # 如果转换失败，使用原始值
                                cell_value = str(cell.value)
                        else:
                            # 普通数字，转换为字符串
                            cell_value = str(cell.value)
                    else:
                        # 对于文本类型，直接使用值
                        cell_value = str(cell.value) if cell.value is not None else ""
                    
                    row_data[column_names[i]] = cell_value
            
            # 只添加非空行
            if any(row_data.values()):
                data_rows.append(row_data)
        
        # 创建DataFrame
        extracted_df = pd.DataFrame(data_rows)
        
        # 转换为JSON
        json_data = extracted_df.to_json(orient='records', indent=2, force_ascii=False)
        
        # 如果需要保存到文件
        if output_json_path:
            with open(output_json_path, 'w', encoding='utf-8') as f:
                f.write(json_data)
            print(f"JSON文件已保存到: {output_json_path}")
        
        wb.close()
        return json_data, extracted_df
        
    except Exception as e:
        print(f"处理文件时出错: {e}")
        import traceback
        traceback.print_exc()
        return None, None


# 示例用法
excel_file = "../storage/1114test.xlsx"  # 替换为你的Excel文件路径
columns_to_extract = ['B', 'D', 'K']  # 要提取的列

json_result, df_result = extract_columns_to_json(
    file_path=excel_file,
    columns_to_extract=columns_to_extract,
    output_json_path="../storage/extracted_data.json"
)

if json_result:
    print("提取的JSON数据:")
    print(json_result)
    print("\n提取的DataFrame:")
    print(df_result.head())

JSON文件已保存到: ../storage/extracted_data.json
提取的JSON数据:
[
  {
    "字段":"客户名称",
    "客观事实数据":"金富科技股份有限公司",
    "251114测试结果":"客户名称是金富科技股份有限公司[1]。"
  },
  {
    "字段":"注册地址 ",
    "客观事实数据":"广东省东莞市厚街镇恒通路10号",
    "251114测试结果":"注册地址为广东省东莞市沙田镇丽海中路43号[1]。"
  },
  {
    "字段":"注册资本(万元)",
    "客观事实数据":"26000万元",
    "251114测试结果":"26000万[1]"
  },
  {
    "字段":"成立时间",
    "客观事实数据":"2001-01-20 00:00:00",
    "251114测试结果":"金富科技股份有限公司的成立日期为2001年01月20日[1]。"
  },
  {
    "字段":"所属行业（大-中-小类） ",
    "客观事实数据":"制造业-塑料制品业-塑料包装箱及容器制造",
    "251114测试结果":"所属行业（大-中-小类）为：制造业（大类）[1]，塑料制品业（中类）[2]，塑料包装箱及容器制造（小类）[3]。"
  },
  {
    "字段":"客户规模及行业地位",
    "客观事实数据":"中型企业",
    "251114测试结果":"公司深耕饮料瓶盖产品市场超过二十年[1]，已成为国内最大的塑料防盗瓶盖供应商之一[2]，并成为华润怡宝、景田、可口可乐、达能等国内知名饮料制造企业的瓶盖主要供应商之一[3]。公司产品广泛应用于饮料、调味品、碳酸饮料、奶饮料等多个领域[4]，行业地位突出。2022年，公司实现营业收入738,068,573.03元，同比增长11.93%[5]，并且前五大客户合计销售金额占年度销售总额的83.30%[6]，显示出其在客户中的重要地位。"
  },
  {
    "字段":"实际控制人",
    "客观事实数据":"陈金培,陈婉如,陈珊珊",
    "251114测试结果":"公司的实际控制人为陈金培、陈婉如和陈珊珊。三人合计控制公司发行后股份的70.16%，并且签署了《

In [8]:
# # 计算BLEU值

# import json
# import math
# import jieba
# import nltk

# # 下载nltk所需的数据（如果还没有下载）
# try:
#     nltk.data.find('tokenizers/punkt')
# except LookupError:
#     print("正在下载nltk的punkt数据...")
#     nltk.download('punkt', quiet=True)
#     print("下载完成")

# from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# def calculate_bleu(reference, candidate):
#     """
#     计算BLEU分数
    
#     参数:
#     reference: 参考文本（客观事实数据）
#     candidate: 候选文本（251114测试结果）
    
#     返回:
#     BLEU分数（0-1之间），如果任一文本为空则返回NaN
#     """
#     # 检查是否为空
#     if not reference or not candidate:
#         return float('nan')
    
#     # 去除空白字符
#     reference = str(reference).strip()
#     candidate = str(candidate).strip()
    
#     if not reference or not candidate:
#         return float('nan')
    
#     # 使用jieba进行中文分词
#     reference_tokens = list(jieba.cut(reference))
#     candidate_tokens = list(jieba.cut(candidate))
    
#     # 使用平滑函数避免零分
#     smoothing = SmoothingFunction().method1
    
#     # 计算BLEU分数（使用1-4 gram）
#     try:
#         bleu_score = sentence_bleu(
#             [reference_tokens], 
#             candidate_tokens,
#             smoothing_function=smoothing
#         )
#         return bleu_score
#     except:
#         return float('nan')

# # 读取JSON数据
# json_file_path = "../storage/extracted_data.json"
# try:
#     with open(json_file_path, 'r', encoding='utf-8') as f:
#         data = json.load(f)
    
#     # 为每条记录计算BLEU值
#     bleu_scores = []
#     for item in data:
#         reference = item.get("客观事实数据", "")
#         candidate = item.get("251114测试结果", "")
        
#         bleu_value = calculate_bleu(reference, candidate)
#         item["BLEU"] = bleu_value
        
#         # 收集有效BLEU值（非NaN）
#         if not math.isnan(bleu_value):
#             bleu_scores.append(bleu_value)
    
#     # 计算有效BLEU值的平均数
#     if bleu_scores:
#         average_bleu = sum(bleu_scores) / len(bleu_scores)
#         print(f"有效BLEU值的平均数: {average_bleu:.4f}")
#         print(f"有效BLEU值数量: {len(bleu_scores)} / {len(data)}")
#     else:
#         print("警告: 没有有效的BLEU值")
#         average_bleu = float('nan')
    
#     # 将NaN值转换为空字符串，以便正确保存到JSON
#     for item in data:
#         if 'BLEU' in item and (math.isnan(item['BLEU']) or item['BLEU'] is None):
#             item['BLEU'] = ""
    
#     # 输出添加BLEU值后的JSON数据
#     output_json = json.dumps(data, ensure_ascii=False, indent=2)
#     print("\n添加BLEU值后的JSON数据（前5条）:")
#     print(json.dumps(data[:5], ensure_ascii=False, indent=2))
    
#     # 保存完整结果
#     output_file = "../storage/extracted_data_with_bleu.json"
#     with open(output_file, 'w', encoding='utf-8') as f:
#         f.write(output_json)
#     print(f"\n完整结果已保存到: {output_file}")
    
# except FileNotFoundError:
#     print(f"错误: 找不到文件 {json_file_path}")
#     print("请先运行上一个代码块生成JSON数据")
# except Exception as e:
#     print(f"处理时出错: {e}")
#     import traceback
#     traceback.print_exc()



## 使用 Ragas 评估答案质量

使用 ragas 库对答案进行事实正确性（Factual Correctness）和语义相似度（Semantic Similarity）评估


In [9]:
# # RAGAS简单测试

OPENAI_BASE_URL = "http://103.154.31.78:20001/compatible-mode/v1"
OPENAI_API_KEY = "sk-1"
MODEL_NAME = "qwen3-max"
from ragas.dataset_schema import SingleTurnSample
from langchain_openai import ChatOpenAI
from ragas import evaluate
from ragas.metrics import FactualCorrectness, SemanticSimilarity
from ragas.llms import LangchainLLMWrapper
# 使用 LangChain ChatOpenAI 创建 LLM 实例（虽然 LangchainLLMWrapper 已弃用，但更兼容 ragas）
qwen_llm = ChatOpenAI(
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_BASE_URL,
    model=MODEL_NAME,
    temperature=0.3,
)

# 使用 LangchainLLMWrapper 包装（虽然已弃用，但更兼容 ragas 的 FactualCorrectness）
evaluator_llm = LangchainLLMWrapper(qwen_llm)

# factual_correctness_F1 = FactualCorrectness(
#     llm=evaluator_llm,
#     mode="F1"  # 使用 F1 模式
# )
# factual_correctness_precision = FactualCorrectness(
#     llm=evaluator_llm,
#     mode="precision"  # 使用 precision 模式
# )
# factual_correctness_recall = FactualCorrectness(
#     llm=evaluator_llm,
#     mode="recall"  # 使用 recall 模式
# )
# samples = []
# samples.append(SingleTurnSample(
#     response="埃菲尔铁塔位于巴黎。",
#     reference="埃菲尔铁塔位于巴黎，高度为1000英尺。"
# )) # should be 0.67(F1)

# samples.append(SingleTurnSample(
#     response="金富科技股份有限公司的注册地址为广东省东莞市厚街镇恒通路10号。",
#     reference="广东省东莞市厚街镇恒通路10号"
# ))

# samples.append(SingleTurnSample(
#     response="金富科技股份有限公司的注册地址是：广东省东莞市厚街镇恒通路10号。",
#     reference="广东省东莞市厚街镇恒通路10号"
# ))

# samples.append(SingleTurnSample(
#     response="金富科技股份有限公司的注册地址是，广东省东莞市厚街镇恒通路10号。",
#     reference="广东省东莞市厚街镇恒通路10号"
# ))

# samples.append(SingleTurnSample(
#     response="金富科技股份有限公司的注册地址是，广东省东莞市厚街镇恒通路10号",
#     reference="广东省东莞市厚街镇恒通路10号"
# ))

# samples.append(SingleTurnSample(
#     response="广东省东莞市厚街镇恒通路10号",
#     reference="广东省东莞市厚街镇恒通路10号"
# ))

# samples.append(SingleTurnSample(
#     response="广东省东莞市厚街镇恒通路10号。",
#     reference="广东省东莞市厚街镇恒通路10号"
# ))

# for sample in samples:
#     result_F1 = await factual_correctness_F1.single_turn_ascore(sample)
#     result_precision = await factual_correctness_precision.single_turn_ascore(sample)
#     result_recall = await factual_correctness_recall.single_turn_ascore(sample)
#     print(f"response: {sample.response}")
#     print(f"reference: {sample.reference}")
#     print(f"FactualCorrectness F1: {result_F1}")
#     print(f"FactualCorrectness Precision: {result_precision}")
#     print(f"FactualCorrectness Recall: {result_recall}")
#     print("="*30)

C:\Users\andya\AppData\Local\Temp\ipykernel_107984\1703634663.py:9: DeprecationWarning: Importing FactualCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import FactualCorrectness
  from ragas.metrics import FactualCorrectness, SemanticSimilarity
C:\Users\andya\AppData\Local\Temp\ipykernel_107984\1703634663.py:9: DeprecationWarning: Importing SemanticSimilarity from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import SemanticSimilarity
  from ragas.metrics import FactualCorrectness, SemanticSimilarity
C:\Users\andya\AppData\Local\Temp\ipykernel_107984\1703634663.py:20: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4

In [10]:
import sys
from pathlib import Path

# 动态将项目根目录加入 sys.path，方便本地开发下 src.xxx 导入
# 使用 get_ipython().magic to get the notebook file location
try:
    notebook_path = get_ipython().run_line_magic('pwd', '')
    project_root = Path(notebook_path).resolve().parents[0]
    if str(project_root) not in sys.path:
        sys.path.insert(0, str(project_root))
except Exception:
    pass

from src.agents.content_generate_agent.utils import apply_prompt_template
from ragas.metrics import DiscreteMetric
from openai import OpenAI
from ragas.llms import llm_factory

# 配置 LLM（使用与之前评估相同的配置）
OPENAI_BASE_URL = "http://103.154.31.78:20001/compatible-mode/v1"
OPENAI_API_KEY = "sk-1"
MODEL_NAME = "qwen3-max"

# 创建 OpenAI 客户端
openai_client = OpenAI(
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_BASE_URL,
)

# 使用 llm_factory 创建 LLM（支持结构化输出）
evaluator_llm = llm_factory(
    model=MODEL_NAME,
    provider="openai",
    client=openai_client,
    temperature=0.3,
)

# 测试数据
response = "金富科技股份有限公司的注册地址是，广东省东莞市厚街镇恒通路10号"
reference = "广东省东莞市厚街镇恒通路10号"

# 读取 prompt 模板文件（ragas 使用 Python format，不是 Jinja2）
from pathlib import Path
# 在 notebook 中，使用 project_root 来定位文件
prompt_file = project_root / "src" / "agents" / "content_generate_agent" / "prompts" / "quality_score.md"
with open(prompt_file, 'r', encoding='utf-8') as f:
    prompt_template = f.read().strip()

# 创建质量评分指标
# 注意：allowed_values 需要是字符串列表，prompt 模板会在 score() 时动态渲染
quality_metric = DiscreteMetric(
    name="quality",
    allowed_values=[str(i) for i in range(1, 6)],  # 1到5分，转换为字符串列表
    prompt=prompt_template,  # ragas 使用 Python format，所以直接使用模板字符串
    # llm=evaluator_llm
)

# 调用 score 方法，传入 llm、response 和 reference 作为关键字参数
# score() 方法会使用 prompt 模板，并将 response 和 reference 参数传入模板
result = quality_metric.score(llm=evaluator_llm, response=response, reference=reference)
print(f"评分结果: {result.value}")
print(f"评分理由: {result.reason}")

评分结果: 5
评分理由: 输入答案完整包含了参考答案中的所有关键信息点：'广东省东莞市厚街镇恒通路10号'。虽然输入答案在前面添加了'金富科技股份有限公司的注册地址是，'这一额外说明，但这并不影响对参考答案信息点的完整覆盖。地址内容完全一致，表述清晰准确，没有遗漏任何细节。因此符合5分（优秀）的标准。


In [ ]:
# 从 0202ground_truth.json 提取 ground_truth（适配新格式）
# 新格式：字段名为 "gt" 而不是 "ground_truth"，"id" 对应 question_id
import json
from pathlib import Path

# 读取 ground_truth 数据（从 0202ground_truth.json）
TEST_VER = "0202"
ground_truth_file = Path(f"../storage/testlog/{TEST_VER}ground_truth.json")
if ground_truth_file.exists():
    with open(ground_truth_file, 'r', encoding='utf-8') as f:
        ground_truth_data = json.load(f)
    
    # 创建 question_id 到 ground_truth 的映射
    # 注意：新格式中字段是 "gt" 而不是 "ground_truth"
    ground_truth_by_id = {}
    for item in ground_truth_data:
        question_id = item.get("id", "").strip()
        # 新格式使用 "gt" 字段
        ground_truth_value = item.get("gt", "").strip()
        if question_id and ground_truth_value:
            ground_truth_by_id[question_id] = ground_truth_value
    
    print(f"从 {ground_truth_file} 读取了 {len(ground_truth_data)} 条问题任务数据")
    print(f"创建了 {len(ground_truth_by_id)} 个 question_id 到 ground_truth 的映射")
    print()
else:
    print(f"⚠️ 警告: 未找到 ground_truth 文件 {ground_truth_file}")
    ground_truth_by_id = {}
    print()


从 ..\storage\testlog\0202ground_truth.json 读取了 43 条问题任务数据
创建了 43 个 question_id 到 ground_truth 的映射



In [12]:
# 使用 Ragas 评估答案质量
import json
import pandas as pd
from pathlib import Path
from datasets import Dataset
from langchain_openai import ChatOpenAI
from langchain_community.embeddings import DashScopeEmbeddings
from ragas import evaluate
# from ragas.metrics import FactualCorrectness, SemanticSimilarity, DiscreteMetric, Faithfulness
from ragas.metrics import DiscreteMetric, Faithfulness
from ragas.llms import LangchainLLMWrapper, llm_factory
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.run_config import RunConfig
from openai import OpenAI

# 配置 ragas 使用阿里云 qwen 模型（LLM）
OPENAI_BASE_URL = "http://103.154.31.78:20001/compatible-mode/v1"
OPENAI_API_KEY = "sk-1"
MODEL_NAME = "qwen3-max"

# OPENAI_BASE_URL = "http://10.192.72.12:9996/v1"
# OPENAI_API_KEY = "1"
# MODEL_NAME = "Qwen3-32B"

# 配置 Embedding 模型（使用 DashScope）
# 参考用户提供的示例
EMBEDDING_API_KEY = "sk-a40d1a6e2e6c42539c67322d7bb39f11"
EMBEDDING_MODEL = "text-embedding-v2"

# 使用 LangChain ChatOpenAI 创建 LLM 实例（虽然 LangchainLLMWrapper 已弃用，但更兼容 ragas）
qwen_llm = ChatOpenAI(
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_BASE_URL,
    model=MODEL_NAME,
    temperature=0.3,
)

# 使用 LangchainLLMWrapper 包装（虽然已弃用，但更兼容 ragas 的 FactualCorrectness）
evaluator_llm = LangchainLLMWrapper(qwen_llm)

# 创建 Faithfulness 指标（不需要 ground_truth，对所有数据都可用）
faithfulness = Faithfulness(llm=evaluator_llm)

# 使用 DashScopeEmbeddings 创建 Embedding 模型实例
dashscope_embeddings = DashScopeEmbeddings(
    model=EMBEDDING_MODEL,
    dashscope_api_key=EMBEDDING_API_KEY
)

# 使用 LangchainEmbeddingsWrapper 包装（ragas 需要）
evaluator_embeddings = LangchainEmbeddingsWrapper(dashscope_embeddings)

# 创建 LangChain LLM 实例
# LLM 和 Embedding 已在上面创建

print(f"已配置 ragas 使用 qwen 模型")
print(f"LLM Base URL: {OPENAI_BASE_URL}")
print(f"LLM Model: {MODEL_NAME}")
print(f"Embedding Provider: DashScope")
print(f"Embedding Model: {EMBEDDING_MODEL}")

# 读取答案数据（从 0202test_answers_with_context.json，上下文已在其中）
TEST_VER = "0202"
answers_file = Path(f"../storage/testlog/{TEST_VER}test_answers_with_context.json")
with open(answers_file, 'r', encoding='utf-8') as f:
    answers_data = json.load(f)

print(f"从 {answers_file} 读取了 {len(answers_data)} 个答案（包含上下文）")
print()

# 创建 question_id 到 context_text 的映射（从答案数据中提取）
# 新格式：上下文已经在答案数据中，字段为 "context_text"（字符串）
question_id_to_context = {}
for answer_item in answers_data:
    question_id = answer_item.get("question_id", "")
    if question_id:
        question_id = question_id.strip()
    
    context_text = answer_item.get("context_text")
    if context_text is not None:
        context_text = str(context_text).strip()
    else:
        context_text = ""
    
    if question_id and context_text:
        question_id_to_context[question_id] = context_text

print(f"从答案数据中提取了 {len(question_id_to_context)} 个问题的上下文")
print()

# 定义函数：从答案数据中获取上下文（适配新格式）
def get_context_for_question(question_id, question_text, answers_data, verbose=True):
    """
    从答案数据中获取上下文（新格式：上下文已在答案数据中）
    
    Args:
        question_id: 问题ID
        question_text: 问题文本（用于备用匹配）
        answers_data: 答案数据列表（包含 context_text 字段）
        verbose: 是否输出警告信息
    
    Returns:
        context_text: 如果找到，返回 context_text 字符串；否则返回 None
    """
    # 方法1: 通过 question_id 匹配
    for answer_item in answers_data:
        item_qid = answer_item.get("question_id", "")
        if item_qid and item_qid.strip() == question_id:
            context_text = answer_item.get("context_text")
            if context_text is not None:
                context_text = str(context_text).strip()
                if context_text:
                    return context_text
    
    # 方法2: 通过 question_text 匹配（备用）
    for answer_item in answers_data:
        item_qtext = answer_item.get("question_text", "")
        if item_qtext and item_qtext.strip() == question_text:
            context_text = answer_item.get("context_text")
            if context_text is not None:
                context_text = str(context_text).strip()
                if context_text:
                    return context_text
    
    if verbose:
        print(f"⚠️ 警告: 问题 '{question_text[:50]}...' (ID: {question_id}) 未找到上下文，跳过 Faithfulness 评估")
    return None

# 定义函数：评估 Faithfulness
async def evaluate_faithfulness_score(question_text, answer_text, context_text):
    """
    评估 Faithfulness 分数（使用自定义实现，替换 ragas 默认指标）
    
    Args:
        question_text: 问题文本
        answer_text: 答案文本
        context_text: 上下文文本（字符串，新格式）或上下文文本列表（旧格式）
    
    Returns:
        faithfulness_score: Faithfulness 分数，如果评估失败返回 -1
    """
    import os, sys
    from pathlib import Path

    original_cwd = os.getcwd()
    project_root = Path(__file__).resolve().parent.parent if '__file__' in globals() else Path(original_cwd).resolve().parent
    try:
        os.chdir(project_root)
        if str(project_root) not in sys.path:
            sys.path.insert(0, str(project_root))
    except Exception as e:
        print(f"⚠️ 切换工作目录失败，将继续使用原目录: {e}")
        project_root = Path(original_cwd)

    try:
        from src.agents.content_generate_agent.evaluate.custom_faithfulness import (
            CustomFaithfulnessEvaluator,
        )
        import asyncio

        # 如果 context_text 是字符串，转换为列表（评估器期望列表格式）
        if isinstance(context_text, str):
            context_texts = [context_text]
        elif isinstance(context_text, list):
            context_texts = context_text
        else:
            context_texts = []

        evaluator = CustomFaithfulnessEvaluator(model_id="model_bailian_max", context_token_limit=110000)
        result = await evaluator.evaluate(question_text, answer_text, context_texts)

        return float(result.faithfulness) if result else -1
    except Exception as e:
        print(f"⚠️ Faithfulness 评估失败: {e}")
        return -1
    finally:
        try:
            os.chdir(original_cwd)
        except Exception:
            pass

# 读取客观事实数据（从 extracted_data.json，即 Cell 0 的结果）
# 这是旧方法，仅在新方法（ground_truth_by_id）不存在或未找到匹配时使用
ground_truth_map = {}
try:
    # 检查新方法是否存在
    if 'ground_truth_by_id' not in locals() and 'ground_truth_by_id' not in globals():
        raise NameError("ground_truth_by_id not found")
    if not ground_truth_by_id:
        raise ValueError("ground_truth_by_id is empty")
except (NameError, ValueError):
    ground_truth_file = Path("../storage/extracted_data.json")
    if ground_truth_file.exists():
        with open(ground_truth_file, 'r', encoding='utf-8') as f:
            ground_truth_data = json.load(f)
        
        print(f"从 {ground_truth_file} 读取了 {len(ground_truth_data)} 条客观事实数据")
        print()
        
        # 创建 question_text 到 ground_truth 的映射
        # 注意：需要处理字段名中的空格和特殊字符
        for item in ground_truth_data:
            field_name = item.get("字段", "").strip()
            ground_truth_value = item.get("客观事实数据", "").strip()
            if field_name and ground_truth_value:
                ground_truth_map[field_name] = ground_truth_value
        
        print(f"创建了 {len(ground_truth_map)} 个字段到客观事实数据的映射")
        print()
    else:
        print(f"⚠️ 警告: 未找到旧方法的 ground_truth 文件 {ground_truth_file}")
        print()
else:
    print("使用新方法（ground_truth_by_id）提取 ground_truth，跳过旧方法")
    print()

# 准备评估数据
evaluation_data = []
for answer_item in answers_data:
    question_id = answer_item.get("question_id", "")
    question_text = answer_item.get("question_text", "").strip()
    answer_raw = answer_item.get("answer_raw", "").strip()
    
    # 通过 question_id 或 question_text 匹配 ground_truth
    # 优先使用新方法（通过 question_id），如果找不到再使用旧方法（通过 question_text）
    ground_truth = ""
    
    # 方法1: 通过 question_id 匹配（新方法，适用于 1218test_answers.json）
    try:
        if 'ground_truth_by_id' in locals() or 'ground_truth_by_id' in globals():
            if question_id in ground_truth_by_id:
                ground_truth = ground_truth_by_id[question_id]
    except (NameError, KeyError):
        pass
    
    # 方法2: 通过 question_text 匹配（旧方法，适用于 extracted_data.json）
    if not ground_truth:
        try:
            if ('ground_truth_map' in locals() or 'ground_truth_map' in globals()) and ground_truth_map:
                # 方法2.1: 精确匹配
                if question_text in ground_truth_map:
                    ground_truth = ground_truth_map[question_text]
                else:
                    # 方法2.2: 去除首尾空格后匹配
                    question_text_stripped = question_text.strip()
                    for key, value in ground_truth_map.items():
                        if key.strip() == question_text_stripped:
                            ground_truth = value
                            break
                    
                    # 方法2.3: 去除所有空白字符后匹配（处理"注册地址 "这种情况）
                    if not ground_truth:
                        question_text_no_space = question_text.replace(" ", "").replace("\t", "").replace("\n", "")
                        for key, value in ground_truth_map.items():
                            key_no_space = key.replace(" ", "").replace("\t", "").replace("\n", "")
                            if key_no_space == question_text_no_space:
                                ground_truth = value
                                break
        except (NameError, KeyError):
            pass
    
    # 如果找到 ground_truth，添加到评估数据中
    if ground_truth:
        evaluation_data.append({
            "question_id": question_id,
            "question_text": question_text,
            "answer_raw": answer_raw,
            "ground_truth": ground_truth
        })
    else:
        # 即使没有 ground_truth，也添加到数据中（后续用 -1 填充指标）
        evaluation_data.append({
            "question_id": question_id,
            "question_text": question_text,
            "answer_raw": answer_raw,
            "ground_truth": ""
        })

print(f"准备评估 {len(evaluation_data)} 条数据")
print(f"其中 {sum(1 for item in evaluation_data if item['ground_truth'])} 条有 ground_truth")
print(f"其中 {sum(1 for item in evaluation_data if not item['ground_truth'])} 条没有 ground_truth")
print()

# 分离有 ground_truth 和没有 ground_truth 的数据
data_with_gt = [item for item in evaluation_data if item['ground_truth']]
data_without_gt = [item for item in evaluation_data if not item['ground_truth']]

# 创建 helpfulness 评估所需的 LLM 和 metric（因为 helpfulness 不需要 ground_truth，对所有数据都可用）
# 这些变量将在有 ground_truth 和没有 ground_truth 的数据评估中都会使用
helpfulness_metric = None
quality_llm = None
try:
    quality_openai_client = OpenAI(
        api_key=OPENAI_API_KEY,
        base_url=OPENAI_BASE_URL,
    )
    quality_llm = llm_factory(
        model=MODEL_NAME,
        provider="openai",
        client=quality_openai_client,
        temperature=0.3,
    )
    # 读取有用性评分 prompt 模板
    try:
        helpfulness_prompt_file = project_root / "src" / "agents" / "content_generate_agent" / "prompts" / "helpfulness_score.md"
    except NameError:
        helpfulness_prompt_file = Path("../src/agents/content_generate_agent/prompts/helpfulness_score.md")
    with open(helpfulness_prompt_file, 'r', encoding='utf-8') as f:
        helpfulness_prompt_template = f.read().strip()
    
    helpfulness_metric = DiscreteMetric(
        name="helpfulness",
        allowed_values=[str(i) for i in range(1, 6)],  # 1到5分
        prompt=helpfulness_prompt_template,
    )
except Exception as e:
    print(f"创建 helpfulness 评估指标时出错: {e}")

# 只对有 ground_truth 的数据进行评估（包括 helpfulness）
if data_with_gt:
    print("开始使用 Ragas 进行评估...")
    
    # 执行评估 - 改为逐条评估以避免并发过多导致超时
    # 初始化变量（在 try 块外，确保在异常情况下也能使用）
    fc_f1_col = None
    # fc_precision_col = None  # 注释掉 precision
    # fc_recall_col = None  # 注释掉 recall
    ss_col = None
    quality_col = None
    helpfulness_col = None
    results_df = None
    all_results = []  # 存储所有逐条评估的结果
    
    try:
        # 创建评估指标，传入自定义 LLM
        # 注释掉 factual_correctness 和 semantic_similarity 指标
        # factual_correctness_F1 = FactualCorrectness(
        #     llm=evaluator_llm,
        #     mode="F1"  # 使用 F1 模式
        # )
        # # 注释掉 precision 和 recall 模式
        # # factual_correctness_precision = FactualCorrectness(
        # #     llm=evaluator_llm,
        # #     mode="precision"  # 使用 precision 模式
        # # )
        # # factual_correctness_recall = FactualCorrectness(
        # #     llm=evaluator_llm,
        # #     mode="recall"  # 使用 recall 模式
        # # )
        
        # 创建质量评分指标（使用与 evaluator_llm 相同的 LLM 配置）
        # 由于 DiscreteMetric 需要支持结构化输出的 LLM，使用 llm_factory 创建（但使用相同的配置）
        # 使用相同的配置创建 OpenAI 客户端
        quality_openai_client = OpenAI(
            api_key=OPENAI_API_KEY,
            base_url=OPENAI_BASE_URL,
        )
        # 使用 llm_factory 创建支持结构化输出的 LLM（使用与 evaluator_llm 相同的配置）
        quality_llm = llm_factory(
            model=MODEL_NAME,
            provider="openai",
            client=quality_openai_client,
            temperature=0.3,
        )
        
        # 读取质量评分 prompt 模板
        # 在 notebook 中，使用相对路径或 project_root
        try:
            # 尝试使用 project_root（如果已定义）
            prompt_file = project_root / "src" / "agents" / "content_generate_agent" / "prompts" / "quality_score.md"
        except NameError:
            # 如果 project_root 未定义，使用相对路径
            prompt_file = Path("../src/agents/content_generate_agent/prompts/quality_score.md")
        with open(prompt_file, 'r', encoding='utf-8') as f:
            quality_prompt_template = f.read().strip()
        
        quality_metric = DiscreteMetric(
            name="quality",
            allowed_values=[str(i) for i in range(1, 6)],  # 1到5分
            prompt=quality_prompt_template,
            # llm=quality_llm  # 使用与 evaluator_llm 相同配置的 LLM（相同的 base_url、api_key、model）
        )
        
        # helpfulness_metric 已在上面创建（对所有数据都可用）
        
        # 注释掉 SemanticSimilarity 指标
        # SemanticSimilarity：使用embedding向量和余弦相似度衡量生成答案与参考答案的语义相似程度
        # SemanticSimilarity 使用 embedding 模型
        # semantic_similarity = SemanticSimilarity(embeddings=evaluator_embeddings)
        
        # Faithfulness 已在外部创建，这里直接使用
        
        print(f"开始逐条评估 {len(data_with_gt)} 条数据（避免并发过多导致超时）...")
        print()
        
        # 逐条评估
        for idx, item in enumerate(data_with_gt, 1):
            try:
                # 为单条数据创建 Dataset
                single_data = {
                    "question": [item["question_text"]],
                    "answer": [item["answer_raw"]],
                    "ground_truth": [item["ground_truth"]]
                }
                single_dataset = Dataset.from_dict(single_data)
                
                # 评估单条数据（使用 ragas evaluate）
                # 注释掉 factual_correctness 和 semantic_similarity 评估
                # result = evaluate(
                #     dataset=single_dataset,
                #     metrics=[factual_correctness_F1, semantic_similarity],  # 注释掉 precision 和 recall
                #     run_config=RunConfig(max_workers=1)  # 单条评估，使用单线程
                # )
                # 创建一个空的结果 DataFrame，因为不再使用 ragas evaluate
                result = None
                
                # 单独评估质量评分（使用 DiscreteMetric）
                quality_result = quality_metric.score(
                    llm=quality_llm,
                    response=item["answer_raw"],
                    reference=item["ground_truth"]
                )
                
                # 单独评估有用性评分（使用 DiscreteMetric）
                helpfulness_result = helpfulness_metric.score(
                    llm=quality_llm,
                    question=item["question_text"],
                    response=item["answer_raw"]
                )
                
                # 评估 Faithfulness（需要匹配上下文）
                faithfulness_score = -1
                question_id = item.get("question_id", "")
                question_text = item["question_text"]
                
                # 获取上下文（从答案数据中获取）
                context_text = get_context_for_question(question_id, question_text, answers_data, verbose=True)
                
                # 如果找到上下文，评估 Faithfulness
                if context_text:
                    faithfulness_score = await evaluate_faithfulness_score(
                        question_text, 
                        item["answer_raw"], 
                        context_text
                    )
                    if faithfulness_score == -1:
                        print(f"⚠️ 第 {idx} 条数据 Faithfulness 评估失败")
                

                # 将质量评分和有用性评分结果添加到 DataFrame
                # 注释掉 factual_correctness 和 semantic_similarity 后，不再使用 result.to_pandas()
                # single_result_df = result.to_pandas()
                # 创建一个新的 DataFrame，只包含需要的列
                single_result_df = pd.DataFrame({
                    "question": [item["question_text"]],
                    "answer": [item["answer_raw"]],
                    "ground_truth": [item["ground_truth"]]
                })
                single_result_df["quality_score"] = [quality_result.value]
                single_result_df["quality_reason"] = [quality_result.reason]
                single_result_df["helpfulness_score"] = [helpfulness_result.value]
                single_result_df["helpfulness_reason"] = [helpfulness_result.reason]
                single_result_df["faithfulness"] = [faithfulness_score]
                
                # 添加到结果列表
                all_results.append(single_result_df)
                
                # 显示进度
                if idx % 1 == 0 or idx == len(data_with_gt):
                    print(f"  已评估 {idx}/{len(data_with_gt)} 条数据...")
                    
            except Exception as single_error:
                print(f"\n⚠️ 第 {idx} 条数据评估失败:")
                print(f"   错误类型: {type(single_error).__name__}")
                print(f"   错误消息: {str(single_error)[:200]}")
                
                # 为失败的数据创建空结果（所有指标为 NaN）
                failed_result = pd.DataFrame({
                    "question": [item["question_text"]],
                    "answer": [item["answer_raw"]],
                    "ground_truth": [item["ground_truth"]],
                    # 注释掉 factual_correctness 和 semantic_similarity
                    # "factual_correctness(mode=precision)": [float('nan')],  # 注释掉 precision
                    # "factual_correctness(mode=recall)": [float('nan')],  # 注释掉 recall
                    # "factual_correctness(mode=F1)": [float('nan')],
                    # "semantic_similarity": [float('nan')],
                    "quality_score": [float('nan')],
                    "quality_reason": [""],
                    "helpfulness_score": [float('nan')],
                    "helpfulness_reason": [""]
                })
                all_results.append(failed_result)
        
        print()
        
        # 合并所有结果
        if all_results:
            results_df = pd.concat(all_results, ignore_index=True)
        else:
            results_df = pd.DataFrame()
        
        # 打印结果列名，用于调试
        if results_df is not None and not results_df.empty:
            print(f"评估结果列名: {list(results_df.columns)}")
            print(f"结果形状: {results_df.shape}")
        else:
            print("⚠️ 警告: 没有成功评估任何数据")
            results_df = pd.DataFrame()
        
        # 检查结果列名（ragas 可能使用不同的列名）
        # 注释掉识别 factual_correctness 和 semantic_similarity 列的代码
        if results_df is not None and not results_df.empty:
            for col in results_df.columns:
                col_lower = col.lower()
                # 注释掉 factual_correctness 和 semantic_similarity 的匹配
                # # 匹配 factual_correctness 列（F1 模式）
                # if 'factual' in col_lower and 'correctness' in col_lower:
                #     if 'mode=f1' in col_lower or '(mode=f1)' in col_lower:
                #         fc_f1_col = col
                #     # 注释掉 precision 和 recall 的匹配
                #     # elif 'mode=precision' in col_lower or '(mode=precision)' in col_lower:
                #     #     fc_precision_col = col
                #     # elif 'mode=recall' in col_lower or '(mode=recall)' in col_lower:
                #     #     fc_recall_col = col
                # # 匹配 semantic_similarity 列
                # if 'semantic' in col_lower and 'similarity' in col_lower:
                #     ss_col = col
                # 匹配 quality_score 列
                if 'quality_score' in col_lower or col == 'quality_score':
                    quality_col = col
                # 匹配 helpfulness_score 列
                if 'helpfulness_score' in col_lower or col == 'helpfulness_score':
                    helpfulness_col = col
        
            # 注释掉查找 factual_correctness 和 semantic_similarity 的代码
            # # 如果找不到，尝试使用默认名称（按顺序匹配）
            # if fc_f1_col is None:
            #     # 尝试查找包含 F1 的列
            #     for col in results_df.columns:
            #         if 'factual' in col.lower() and 'correctness' in col.lower() and ('f1' in col.lower() or 'mode=f1' in col.lower()):
            #             fc_f1_col = col
            #             break
            # # 注释掉 precision 和 recall 的查找
            # # if fc_precision_col is None:
            # #     for col in results_df.columns:
            # #         if 'factual' in col.lower() and 'correctness' in col.lower() and 'precision' in col.lower():
            # #             fc_precision_col = col
            # #             break
            # # if fc_recall_col is None:
            # #     for col in results_df.columns:
            # #         if 'factual' in col.lower() and 'correctness' in col.lower() and 'recall' in col.lower():
            # #             fc_recall_col = col
            # #             break
            # 
            # if ss_col is None:
            #     ss_col = 'semantic_similarity' if 'semantic_similarity' in results_df.columns else None
            
            if quality_col is None:
                quality_col = 'quality_score' if 'quality_score' in results_df.columns else None
            
            if helpfulness_col is None:
                helpfulness_col = 'helpfulness_score' if 'helpfulness_score' in results_df.columns else None
            
            print(f"使用列名:")
            # 注释掉 factual_correctness 和 semantic_similarity 的打印
            # print(f"  factual_correctness_F1: {fc_f1_col}")
            # # print(f"  factual_correctness_precision: {fc_precision_col}")  # 注释掉
            # # print(f"  factual_correctness_recall: {fc_recall_col}")  # 注释掉
            # print(f"  semantic_similarity: {ss_col}")
            print(f"  quality_score: {quality_col}")
            print(f"  helpfulness_score: {helpfulness_col}")
            print()
            
            # 注释掉打印 factual_correctness 和 semantic_similarity 的代码
            # # 打印前几行的实际值，用于调试
            # if fc_f1_col and fc_f1_col in results_df.columns:
            #     print(f"\n前3行的 {fc_f1_col} 值:")
            #     print(results_df[fc_f1_col].head(3).tolist())
            # # 注释掉 precision 和 recall 的打印
            # # if fc_precision_col and fc_precision_col in results_df.columns:
            # #     print(f"\n前3行的 {fc_precision_col} 值:")
            # #     print(results_df[fc_precision_col].head(3).tolist())
            # # if fc_recall_col and fc_recall_col in results_df.columns:
            # #     print(f"\n前3行的 {fc_recall_col} 值:")
            # #     print(results_df[fc_recall_col].head(3).tolist())
            # if ss_col and ss_col in results_df.columns:
            #     print(f"\n前3行的 {ss_col} 值:")
            #     print(results_df[ss_col].head(3).tolist())
            if quality_col and quality_col in results_df.columns:
                print(f"\n前3行的 {quality_col} 值:")
                print(results_df[quality_col].head(3).tolist())
            if helpfulness_col and helpfulness_col in results_df.columns:
                print(f"\n前3行的 {helpfulness_col} 值:")
                print(results_df[helpfulness_col].head(3).tolist())
            print()
        else:
            print("⚠️ 警告: 结果为空，无法识别列名")
        
        # 将评估结果添加到数据中
        if results_df is not None and not results_df.empty:
            for i, item in enumerate(data_with_gt):
                try:
                    # 注释掉提取 factual_correctness 和 semantic_similarity 结果的代码
                    # # 提取 F1 模式的结果
                    # if results_df is not None and fc_f1_col and fc_f1_col in results_df.columns:
                    #     fc_f1_value = results_df.iloc[i][fc_f1_col]
                    #     item["factual_correctness_F1"] = float(fc_f1_value) if not pd.isna(fc_f1_value) else -1
                    # else:
                    #     item["factual_correctness_F1"] = -1
                    # 
                    # # 注释掉 precision 和 recall 的结果提取
                    # # # 提取 precision 模式的结果
                    # # if results_df is not None and fc_precision_col and fc_precision_col in results_df.columns:
                    # #     fc_precision_value = results_df.iloc[i][fc_precision_col]
                    # #     item["factual_correctness_precision"] = float(fc_precision_value) if not pd.isna(fc_precision_value) else -1
                    # # else:
                    # #     item["factual_correctness_precision"] = -1
                    # # 
                    # # # 提取 recall 模式的结果
                    # # if results_df is not None and fc_recall_col and fc_recall_col in results_df.columns:
                    # #     fc_recall_value = results_df.iloc[i][fc_recall_col]
                    # #     item["factual_correctness_recall"] = float(fc_recall_value) if not pd.isna(fc_recall_value) else -1
                    # # else:
                    # #     item["factual_correctness_recall"] = -1
                    # 
                    # # 提取 semantic_similarity 结果
                    # if results_df is not None and ss_col and ss_col in results_df.columns:
                    #     ss_value = results_df.iloc[i][ss_col]
                    #     item["semantic_similarity"] = float(ss_value) if not pd.isna(ss_value) else -1
                    # else:
                    #     item["semantic_similarity"] = -1
                    
                    # 直接设置为 -1，因为不再评估这些指标
                    item["factual_correctness_F1"] = -1
                    item["semantic_similarity"] = -1
                    
                    # 提取 quality_score 结果
                    if results_df is not None and quality_col and quality_col in results_df.columns:
                        quality_value = results_df.iloc[i][quality_col]
                        # quality_score 是字符串（1-10），需要转换为整数
                        try:
                            item["quality_score"] = int(quality_value) if not pd.isna(quality_value) and str(quality_value).strip() else -1
                        except (ValueError, TypeError):
                            item["quality_score"] = -1
                    else:
                        item["quality_score"] = -1
                    
                    # 提取 quality_reason（如果存在）
                    if results_df is not None and 'quality_reason' in results_df.columns:
                        quality_reason_value = results_df.iloc[i]['quality_reason']
                        item["quality_reason"] = str(quality_reason_value) if not pd.isna(quality_reason_value) else ""
                    else:
                        item["quality_reason"] = ""
                    
                    # 提取 helpfulness_score 结果
                    if results_df is not None and helpfulness_col and helpfulness_col in results_df.columns:
                        helpfulness_value = results_df.iloc[i][helpfulness_col]
                        # helpfulness_score 是字符串（1-5），需要转换为整数
                        try:
                            item["helpfulness_score"] = int(helpfulness_value) if not pd.isna(helpfulness_value) and str(helpfulness_value).strip() else -1
                        except (ValueError, TypeError):
                            item["helpfulness_score"] = -1
                    else:
                        item["helpfulness_score"] = -1
                    
                    # 提取 helpfulness_reason（如果存在）
                    if results_df is not None and 'helpfulness_reason' in results_df.columns:
                        helpfulness_reason_value = results_df.iloc[i]['helpfulness_reason']
                        item["helpfulness_reason"] = str(helpfulness_reason_value) if not pd.isna(helpfulness_reason_value) else ""
                    else:
                        item["helpfulness_reason"] = ""
                    
                    # 提取 faithfulness 结果
                    if results_df is not None and 'faithfulness' in results_df.columns:
                        faithfulness_value = results_df.iloc[i]['faithfulness']
                        item["faithfulness"] = float(faithfulness_value) if not pd.isna(faithfulness_value) else -1
                    else:
                        item["faithfulness"] = -1
                except Exception as e:
                    print(f"处理第 {i} 条数据时出错: {e}")
                    # 注释掉 factual_correctness 和 semantic_similarity
                    item["factual_correctness_F1"] = -1
                    # item["factual_correctness_precision"] = -1  # 注释掉
                    # item["factual_correctness_recall"] = -1  # 注释掉
                    item["semantic_similarity"] = -1
                    item["quality_score"] = -1
                    item["quality_reason"] = ""
                    item["helpfulness_score"] = -1
                    item["helpfulness_reason"] = ""
                    item["faithfulness"] = -1
        else:
            # 如果结果为空，所有指标设为 -1
            for item in data_with_gt:
                # 注释掉 factual_correctness 和 semantic_similarity
                item["factual_correctness_F1"] = -1
                # item["factual_correctness_precision"] = -1  # 注释掉
                # item["factual_correctness_recall"] = -1  # 注释掉
                item["semantic_similarity"] = -1
                item["quality_score"] = -1
                item["quality_reason"] = ""
                item["helpfulness_score"] = -1
                item["helpfulness_reason"] = ""
                item["faithfulness"] = -1
        
        print("评估完成！")
        if results_df is not None:
            # 注释掉计算 factual_correctness 和 semantic_similarity 平均值的代码
            # if fc_f1_col and fc_f1_col in results_df.columns:
            #     valid_fc_f1 = results_df[fc_f1_col].dropna()
            #     if len(valid_fc_f1) > 0:
            #         print(f"平均事实正确性(F1): {valid_fc_f1.mean():.4f} (有效值: {len(valid_fc_f1)}/{len(results_df)})")
            # # 注释掉 precision 和 recall 的平均值计算
            # # if fc_precision_col and fc_precision_col in results_df.columns:
            # #     valid_fc_precision = results_df[fc_precision_col].dropna()
            # #     if len(valid_fc_precision) > 0:
            # #         print(f"平均事实正确性(Precision): {valid_fc_precision.mean():.4f} (有效值: {len(valid_fc_precision)}/{len(results_df)})")
            # # if fc_recall_col and fc_recall_col in results_df.columns:
            # #     valid_fc_recall = results_df[fc_recall_col].dropna()
            # #     if len(valid_fc_recall) > 0:
            # #         print(f"平均事实正确性(Recall): {valid_fc_recall.mean():.4f} (有效值: {len(valid_fc_recall)}/{len(results_df)})")
            # if ss_col and ss_col in results_df.columns:
            #     valid_ss = results_df[ss_col].dropna()
            #     if len(valid_ss) > 0:
            #         print(f"平均语义相似度: {valid_ss.mean():.4f} (有效值: {len(valid_ss)}/{len(results_df)})")
            if quality_col and quality_col in results_df.columns:
                valid_quality = results_df[quality_col].dropna()
                if len(valid_quality) > 0:
                    # quality_score 是字符串，需要转换为数值计算平均值
                    try:
                        quality_numeric = valid_quality.astype(str).str.strip().replace('', pd.NA).dropna()
                        quality_numeric = pd.to_numeric(quality_numeric, errors='coerce').dropna()
                        if len(quality_numeric) > 0:
                            print(f"平均质量评分: {quality_numeric.mean():.2f} (有效值: {len(quality_numeric)}/{len(results_df)})")
                    except Exception as e:
                        print(f"计算平均质量评分时出错: {e}")
            if helpfulness_col and helpfulness_col in results_df.columns:
                valid_helpfulness = results_df[helpfulness_col].dropna()
                if len(valid_helpfulness) > 0:
                    # helpfulness_score 是字符串，需要转换为数值计算平均值
                    try:
                        helpfulness_numeric = valid_helpfulness.astype(str).str.strip().replace('', pd.NA).dropna()
                        helpfulness_numeric = pd.to_numeric(helpfulness_numeric, errors='coerce').dropna()
                        if len(helpfulness_numeric) > 0:
                            print(f"平均有用性评分: {helpfulness_numeric.mean():.2f} (有效值: {len(helpfulness_numeric)}/{len(results_df)})")
                    except Exception as e:
                        print(f"计算平均有用性评分时出错: {e}")
            if 'faithfulness' in results_df.columns:
                valid_faithfulness = results_df['faithfulness'].dropna()
                if len(valid_faithfulness) > 0:
                    # faithfulness 是浮点数，直接计算平均值
                    try:
                        faithfulness_numeric = pd.to_numeric(valid_faithfulness, errors='coerce').dropna()
                        if len(faithfulness_numeric) > 0:
                            print(f"平均忠实度: {faithfulness_numeric.mean():.4f} (有效值: {len(faithfulness_numeric)}/{len(results_df)})")
                    except Exception as e:
                        print(f"计算平均忠实度时出错: {e}")
        print()
        
    except Exception as e:
        print(f"评估过程中出错: {e}")
        import traceback
        traceback.print_exc()
        # 如果评估失败，将所有指标设为 -1
        for item in data_with_gt:
            # 注释掉 factual_correctness 和 semantic_similarity
            item["factual_correctness_F1"] = -1
            # item["factual_correctness_precision"] = -1  # 注释掉
            # item["factual_correctness_recall"] = -1  # 注释掉
            item["semantic_similarity"] = -1
            item["quality_score"] = -1
            item["quality_reason"] = ""
            item["helpfulness_score"] = -1
            item["helpfulness_reason"] = ""
            item["faithfulness"] = -1
else:
    print("没有找到有 ground_truth 的数据，跳过需要 ground_truth 的评估")
    print()

# 对于没有 ground_truth 的数据，用 -1 填充需要 ground_truth 的指标，但评估 helpfulness 和 faithfulness
if data_without_gt:
    print(f"开始评估 {len(data_without_gt)} 条没有 ground_truth 的数据的 Helpfulness 和 Faithfulness...")
    print()
    
    # faithfulness 已在外部创建，这里直接使用
    
    if helpfulness_metric and quality_llm:
        for idx, item in enumerate(data_without_gt, 1):
            try:
                # 评估 helpfulness（不需要 ground_truth）
                helpfulness_result = helpfulness_metric.score(
                    llm=quality_llm,
                    question=item["question_text"],
                    response=item["answer_raw"]
                )
                item["helpfulness_score"] = int(helpfulness_result.value) if helpfulness_result.value else -1
                item["helpfulness_reason"] = helpfulness_result.reason if helpfulness_result.reason else ""
                
                # 评估 Faithfulness（需要匹配上下文）
                if faithfulness:
                    question_id = item.get("question_id", "")
                    question_text = item["question_text"]
                    
                    # 获取上下文（从答案数据中获取，不输出警告信息，因为数据量可能较大）
                    context_text = get_context_for_question(question_id, question_text, answers_data, verbose=False)
                    
                    # 如果找到上下文，评估 Faithfulness
                    if context_text:
                        item["faithfulness"] = await evaluate_faithfulness_score(
                            question_text, 
                            item["answer_raw"], 
                            context_text
                        )
                    else:
                        item["faithfulness"] = -1
                else:
                    item["faithfulness"] = -1
                
                if idx % 10 == 0 or idx == len(data_without_gt):
                    print(f"  已评估 {idx}/{len(data_without_gt)} 条数据的 Helpfulness 和 Faithfulness...")
            except Exception as e:
                print(f"⚠️ 第 {idx} 条数据评估失败: {e}")
                item["helpfulness_score"] = -1
                item["helpfulness_reason"] = ""
                item["faithfulness"] = -1
        print()
    else:
        print("⚠️ 警告: 无法创建 Helpfulness 评估指标，跳过评估")
        print()
    
    # 对于没有 ground_truth 的数据，用 -1 填充需要 ground_truth 的指标
    for item in data_without_gt:
        # 注释掉 factual_correctness 和 semantic_similarity
        item["factual_correctness_F1"] = -1
        # item["factual_correctness_precision"] = -1  # 注释掉
        # item["factual_correctness_recall"] = -1  # 注释掉
        item["semantic_similarity"] = -1
        item["quality_score"] = -1
        item["quality_reason"] = ""
        # helpfulness_score 和 helpfulness_reason 已在上面评估或设为 -1
        if "helpfulness_score" not in item:
            item["helpfulness_score"] = -1
        if "helpfulness_reason" not in item:
            item["helpfulness_reason"] = ""
        # faithfulness 也需要评估（如果有上下文的话）
        if "faithfulness" not in item:
            item["faithfulness"] = -1

# 合并所有数据（保持原始顺序）
final_results = []
# 先添加有 ground_truth 的（已评估）
for item in data_with_gt:
    final_results.append(item)
# 再添加没有 ground_truth 的（指标为 -1）
for item in data_without_gt:
    final_results.append(item)

# 按 question_id 排序（可选）
final_results.sort(key=lambda x: x.get("question_id", ""))

# 输出结果
print(f"=== 评估结果 ===")
print(f"共 {len(final_results)} 条数据")
print()

# 显示前5条结果
print("前5条结果示例:")
for i, item in enumerate(final_results[:5], 1):
    print(f"\n{i}. question_id: {item['question_id']}")
    print(f"   question_text: {item['question_text']}")
    print(f"   answer_raw: {item['answer_raw'][:100]}..." if len(item['answer_raw']) > 100 else f"   answer_raw: {item['answer_raw']}")
    print(f"   ground_truth: {item['ground_truth'][:100]}..." if len(item['ground_truth']) > 100 else f"   ground_truth: {item['ground_truth']}")
    # 注释掉打印 factual_correctness 和 semantic_similarity
    # print(f"   factual_correctness_F1: {item.get('factual_correctness_F1', -1)}")
    # # print(f"   factual_correctness_precision: {item.get('factual_correctness_precision', -1)}")  # 注释掉
    # # print(f"   factual_correctness_recall: {item.get('factual_correctness_recall', -1)}")  # 注释掉
    # print(f"   semantic_similarity: {item.get('semantic_similarity', -1)}")
    print(f"   quality_score: {item.get('quality_score', -1)}")
    print(f"   quality_reason: {item.get('quality_reason', '')[:100]}..." if len(item.get('quality_reason', '')) > 100 else f"   quality_reason: {item.get('quality_reason', '')}")
    print(f"   helpfulness_score: {item.get('helpfulness_score', -1)}")
    print(f"   helpfulness_reason: {item.get('helpfulness_reason', '')[:100]}..." if len(item.get('helpfulness_reason', '')) > 100 else f"   helpfulness_reason: {item.get('helpfulness_reason', '')}")
    print(f"   faithfulness: {item.get('faithfulness', -1)}")

# 保存为 JSON 文件
output_file = Path(f"../storage/testlog/{TEST_VER}_test_ragas_evaluation.json")
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(final_results, f, ensure_ascii=False, indent=2)

print()
print(f"=== 结果已保存 ===")
print(f"保存路径: {output_file}")
print(f"共保存 {len(final_results)} 条评估结果")



已配置 ragas 使用 qwen 模型
LLM Base URL: http://103.154.31.78:20001/compatible-mode/v1
LLM Model: qwen3-max
Embedding Provider: DashScope
Embedding Model: text-embedding-v2
从 ..\storage\testlog\0202test_answers_with_context.json 读取了 75 个答案（包含上下文）

从答案数据中提取了 49 个问题的上下文

使用新方法（ground_truth_by_id）提取 ground_truth，跳过旧方法

准备评估 75 条数据
其中 43 条有 ground_truth
其中 32 条没有 ground_truth



C:\Users\andya\AppData\Local\Temp\ipykernel_107984\2792005516.py:10: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import DiscreteMetric, Faithfulness
C:\Users\andya\AppData\Local\Temp\ipykernel_107984\2792005516.py:39: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(qwen_llm)
C:\Users\andya\AppData\Local\Temp\ipykernel_107984\2792005516.py:51: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client)

开始使用 Ragas 进行评估...
开始逐条评估 43 条数据（避免并发过多导致超时）...



2026-02-03 15:24:10.276 | ERROR    | src.utils.langfuse_tracing:__init__:190 - Langfuse auth_check 异常，tracing功能将被禁用: 'str' object cannot be interpreted as an integer
2026-02-03 15:24:10.277 | WARNING  | src.utils.langfuse_tracing:__init__:195 - Langfuse Tracing 已禁用 (配置不完整、鉴权失败或库未安装)


  已评估 1/43 条数据...
  已评估 2/43 条数据...
  已评估 3/43 条数据...
  已评估 4/43 条数据...
  已评估 5/43 条数据...
  已评估 6/43 条数据...
  已评估 7/43 条数据...
  已评估 8/43 条数据...
  已评估 9/43 条数据...
  已评估 10/43 条数据...
  已评估 11/43 条数据...
  已评估 12/43 条数据...
  已评估 13/43 条数据...
  已评估 14/43 条数据...
  已评估 15/43 条数据...
  已评估 16/43 条数据...
  已评估 17/43 条数据...
  已评估 18/43 条数据...
  已评估 19/43 条数据...
  已评估 20/43 条数据...
  已评估 21/43 条数据...
  已评估 22/43 条数据...
  已评估 23/43 条数据...
  已评估 24/43 条数据...
  已评估 25/43 条数据...
  已评估 26/43 条数据...
  已评估 27/43 条数据...
  已评估 28/43 条数据...
  已评估 29/43 条数据...
  已评估 30/43 条数据...
  已评估 31/43 条数据...
  已评估 32/43 条数据...
  已评估 33/43 条数据...
  已评估 34/43 条数据...
  已评估 35/43 条数据...
  已评估 36/43 条数据...
  已评估 37/43 条数据...
  已评估 38/43 条数据...
  已评估 39/43 条数据...
  已评估 40/43 条数据...
  已评估 41/43 条数据...
  已评估 42/43 条数据...
  已评估 43/43 条数据...

评估结果列名: ['question', 'answer', 'ground_truth', 'quality_score', 'quality_reason', 'helpfulness_score', 'helpfulness_reason', 'faithfulness']
结果形状: (43, 8)
使用列名:
  quality_score: quality_score

In [13]:
# 将评估结果 JSON 文件转换为 CSV
import json
import pandas as pd
from pathlib import Path

# 读取 JSON 文件
json_file = Path(f"../storage/testlog/{TEST_VER}_test_ragas_evaluation.json")
with open(json_file, 'r', encoding='utf-8') as f:
    data = json.load(f)

# 转换为 DataFrame
df = pd.DataFrame(data)

# 保存为 CSV
csv_file = Path(f"../storage/testlog/{TEST_VER}_test_ragas_evaluation.csv")
df.to_csv(csv_file, index=False, encoding='utf-8-sig')

print(f"已成功转换 {len(df)} 条数据")
print(f"CSV 文件已保存到: {csv_file}")
print(f"\n数据列: {list(df.columns)}")
print(f"\n前3行数据:")
print(df.head(3))



已成功转换 75 条数据
CSV 文件已保存到: ..\storage\testlog\0202_test_ragas_evaluation.csv

数据列: ['question_id', 'question_text', 'answer_raw', 'ground_truth', 'factual_correctness_F1', 'semantic_similarity', 'quality_score', 'quality_reason', 'helpfulness_score', 'helpfulness_reason', 'faithfulness']

前3行数据:
  question_id                                      question_text  \
0         q_1  需要填写提交该授信初审报告的呈报单位名称，通常为分行或分支机构的全称，应根据实际负责该业务的...   
1        q_10  判断该公司是否属于政府融资平台（包括监管类或监测类），需根据企业股权结构、实际控制人、融资用...   
2        q_11  判断该公司是否为银行的关联方，需根据企业股权结构、管理层关联关系、实际控制人信息等进行核实，...   

                                          answer_raw ground_truth  \
0  提交该授信初审报告的呈报单位名称应为北京银行脱敏B分行，因为该报告由北京银行脱敏B分行开展尽...  北京银行脱敏 B 分行   
1  该公司不属于政府融资平台。根据股权结构，中国铁建股份有限公司的控股股东为中国铁道建筑集团有限...            否   
2  该公司是银行的关联方。根据检索到的上下文，铁建财务为该公司的控股子公司，且该公司与控股股东中...            否   

   factual_correctness_F1  semantic_similarity  quality_score  \
0                      -1                   -1              5   
1                      -1